In [1]:
import pandas as pd
import numpy as np

# Load the 1-minute data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print("="*100)
print("5-MINUTE ORB (9:30-9:35) - COMPREHENSIVE RETEST ANALYSIS")
print("="*100)
print(f"Analyzing {len(dates)-1} trading days")
print("Matching your existing orb_935_direction and orb_935_first_break features\n")

# ============================================================
# CALCULATE RETEST DATA FOR 5-MINUTE ORB ONLY
# ============================================================

retest_data = []

for idx, date in enumerate(dates[1:], 1):
    if idx % 500 == 0:
        print(f"  Processing day {idx}...")
    
    actual_date = date + pd.Timedelta(hours=7)
    
    # ========================================
    # 5-MINUTE ORB (9:30-9:35) - YOUR PRIMARY TRADING VEHICLE
    # ========================================
    
    orb_start = actual_date.replace(hour=9, minute=30, second=0)
    orb_end = actual_date.replace(hour=9, minute=35, second=0)
    orb_data = df.loc[orb_start:orb_end]
    
    if len(orb_data) < 3:
        continue
    
    orb_high = orb_data['High'].max()
    orb_low = orb_data['Low'].min()
    orb_open = orb_data['Open'].iloc[0]
    orb_close = orb_data['Close'].iloc[-1]
    orb_midpoint = (orb_high + orb_low) / 2
    orb_range = orb_high - orb_low
    orb_range_pct = (orb_range / orb_low) * 100
    orb_is_bullish = orb_close > orb_open
    
    # Rest of day
    rest_start = orb_end
    rest_end = actual_date.replace(hour=16, minute=0, second=0)
    rest_of_day = df.loc[rest_start:rest_end]
    
    if rest_of_day.empty:
        continue
    
    # NY session outcome (matching YOUR definition)
    ny_session_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_session_end = actual_date.replace(hour=16, minute=0, second=0)
    ny_session = df.loc[ny_session_start:ny_session_end]
    ny_open = ny_session['Open'].iloc[0]
    ny_close = ny_session['Close'].iloc[-1]
    ny_is_bullish = ny_close > ny_open
    
    # ========================================
    # BREAKOUT DETECTION
    # ========================================
    
    high_broken = False
    low_broken = False
    high_break_time = None
    low_break_time = None
    first_break_direction = None
    
    for idx_row, row in rest_of_day.iterrows():
        if not high_broken and row['High'] > orb_high:
            high_broken = True
            high_break_time = idx_row
            if first_break_direction is None:
                first_break_direction = 'high'
        
        if not low_broken and row['Low'] < orb_low:
            low_broken = True
            low_break_time = idx_row
            if first_break_direction is None:
                first_break_direction = 'low'
    
    # Breakout timing
    high_break_minutes = (high_break_time - orb_end).total_seconds() / 60 if high_break_time else None
    low_break_minutes = (low_break_time - orb_end).total_seconds() / 60 if low_break_time else None
    
    # ========================================
    # AFTER HIGH BREAKOUT - RETEST ANALYSIS
    # ========================================
    
    high_retest_orb_high = False
    high_retest_midpoint = False
    high_retest_orb_low = False
    
    high_orb_high_continues = None
    high_midpoint_continues = None
    high_orb_low_continues = None
    
    high_orb_high_retest_minutes = None
    high_midpoint_retest_minutes = None
    high_orb_low_retest_minutes = None
    
    high_orb_high_mfe = None
    high_orb_high_mae = None
    high_midpoint_mfe = None
    high_midpoint_mae = None
    high_orb_low_mfe = None
    high_orb_low_mae = None
    
    if high_broken and high_break_time is not None:
        data_after_high = rest_of_day.loc[high_break_time:rest_end]
        
        # ORB High retest (shallow pullback)
        for i in range(len(data_after_high)):
            bar = data_after_high.iloc[i]
            tolerance = orb_high * 0.0002
            
            if abs(bar['Low'] - orb_high) < tolerance or (bar['Low'] <= orb_high <= bar['High']):
                high_retest_orb_high = True
                high_orb_high_retest_minutes = (data_after_high.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_high.iloc[i:]
                if len(data_after_retest) > 0:
                    high_orb_high_mfe = ((data_after_retest['High'].max() - orb_high) / orb_high) * 100
                    high_orb_high_mae = ((orb_high - data_after_retest['Low'].min()) / orb_high) * 100
                
                # Continuation check (next 5 bars)
                if i + 5 < len(data_after_high):
                    next_5 = data_after_high.iloc[i+1:i+6]
                    if next_5['High'].max() > orb_high * 1.0005:
                        high_orb_high_continues = True
                    elif next_5['Low'].min() < orb_high * 0.9995:
                        high_orb_high_continues = False
                break
        
        # Midpoint retest (moderate pullback)
        for i in range(len(data_after_high)):
            bar = data_after_high.iloc[i]
            tolerance = orb_midpoint * 0.0002
            
            if abs(bar['Low'] - orb_midpoint) < tolerance or (bar['Low'] <= orb_midpoint <= bar['High']):
                high_retest_midpoint = True
                high_midpoint_retest_minutes = (data_after_high.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_high.iloc[i:]
                if len(data_after_retest) > 0:
                    high_midpoint_mfe = ((data_after_retest['High'].max() - orb_high) / orb_high) * 100
                    high_midpoint_mae = ((orb_high - data_after_retest['Low'].min()) / orb_high) * 100
                
                # Continuation check
                if i + 5 < len(data_after_high):
                    next_5 = data_after_high.iloc[i+1:i+6]
                    if next_5['High'].max() > orb_high * 1.0005:
                        high_midpoint_continues = True
                    elif next_5['Low'].min() < orb_midpoint * 0.9995:
                        high_midpoint_continues = False
                break
        
        # ORB Low retest (deep pullback)
        for i in range(len(data_after_high)):
            bar = data_after_high.iloc[i]
            tolerance = orb_low * 0.0002
            
            if abs(bar['Low'] - orb_low) < tolerance or (bar['Low'] <= orb_low <= bar['High']):
                high_retest_orb_low = True
                high_orb_low_retest_minutes = (data_after_high.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_high.iloc[i:]
                if len(data_after_retest) > 0:
                    high_orb_low_mfe = ((data_after_retest['High'].max() - orb_high) / orb_high) * 100
                    high_orb_low_mae = ((orb_high - data_after_retest['Low'].min()) / orb_high) * 100
                
                # Continuation check
                if i + 5 < len(data_after_high):
                    next_5 = data_after_high.iloc[i+1:i+6]
                    if next_5['High'].max() > orb_high * 1.0005:
                        high_orb_low_continues = True
                    elif next_5['Low'].min() < orb_low * 0.9995:
                        high_orb_low_continues = False
                break
    
    # ========================================
    # AFTER LOW BREAKOUT - RETEST ANALYSIS
    # ========================================
    
    low_retest_orb_low = False
    low_retest_midpoint = False
    low_retest_orb_high = False
    
    low_orb_low_continues = None
    low_midpoint_continues = None
    low_orb_high_continues = None
    
    low_orb_low_retest_minutes = None
    low_midpoint_retest_minutes = None
    low_orb_high_retest_minutes = None
    
    low_orb_low_mfe = None
    low_orb_low_mae = None
    low_midpoint_mfe = None
    low_midpoint_mae = None
    low_orb_high_mfe = None
    low_orb_high_mae = None
    
    if low_broken and low_break_time is not None:
        data_after_low = rest_of_day.loc[low_break_time:rest_end]
        
        # ORB Low retest (shallow pullback)
        for i in range(len(data_after_low)):
            bar = data_after_low.iloc[i]
            tolerance = orb_low * 0.0002
            
            if abs(bar['High'] - orb_low) < tolerance or (bar['Low'] <= orb_low <= bar['High']):
                low_retest_orb_low = True
                low_orb_low_retest_minutes = (data_after_low.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_low.iloc[i:]
                if len(data_after_retest) > 0:
                    low_orb_low_mfe = ((orb_low - data_after_retest['Low'].min()) / orb_low) * 100
                    low_orb_low_mae = ((data_after_retest['High'].max() - orb_low) / orb_low) * 100
                
                # Continuation check
                if i + 5 < len(data_after_low):
                    next_5 = data_after_low.iloc[i+1:i+6]
                    if next_5['Low'].min() < orb_low * 0.9995:
                        low_orb_low_continues = True
                    elif next_5['High'].max() > orb_low * 1.0005:
                        low_orb_low_continues = False
                break
        
        # Midpoint retest (moderate pullback)
        for i in range(len(data_after_low)):
            bar = data_after_low.iloc[i]
            tolerance = orb_midpoint * 0.0002
            
            if abs(bar['High'] - orb_midpoint) < tolerance or (bar['Low'] <= orb_midpoint <= bar['High']):
                low_retest_midpoint = True
                low_midpoint_retest_minutes = (data_after_low.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_low.iloc[i:]
                if len(data_after_retest) > 0:
                    low_midpoint_mfe = ((orb_low - data_after_retest['Low'].min()) / orb_low) * 100
                    low_midpoint_mae = ((data_after_retest['High'].max() - orb_low) / orb_low) * 100
                
                # Continuation check
                if i + 5 < len(data_after_low):
                    next_5 = data_after_low.iloc[i+1:i+6]
                    if next_5['Low'].min() < orb_low * 0.9995:
                        low_midpoint_continues = True
                    elif next_5['High'].max() > orb_midpoint * 1.0005:
                        low_midpoint_continues = False
                break
        
        # ORB High retest (deep pullback)
        for i in range(len(data_after_low)):
            bar = data_after_low.iloc[i]
            tolerance = orb_high * 0.0002
            
            if abs(bar['High'] - orb_high) < tolerance or (bar['Low'] <= orb_high <= bar['High']):
                low_retest_orb_high = True
                low_orb_high_retest_minutes = (data_after_low.index[i] - orb_end).total_seconds() / 60
                
                # MFE/MAE from retest point
                data_after_retest = data_after_low.iloc[i:]
                if len(data_after_retest) > 0:
                    low_orb_high_mfe = ((orb_low - data_after_retest['Low'].min()) / orb_low) * 100
                    low_orb_high_mae = ((data_after_retest['High'].max() - orb_low) / orb_low) * 100
                
                # Continuation check
                if i + 5 < len(data_after_low):
                    next_5 = data_after_low.iloc[i+1:i+6]
                    if next_5['Low'].min() < orb_low * 0.9995:
                        low_orb_high_continues = True
                    elif next_5['High'].max() > orb_high * 1.0005:
                        low_orb_high_continues = False
                break
    
    # Store data
    retest_data.append({
        'date': actual_date.date(),
        'orb_935_high': orb_high,
        'orb_935_low': orb_low,
        'orb_935_midpoint': orb_midpoint,
        'orb_935_range': orb_range,
        'orb_935_range_pct': orb_range_pct,
        'orb_935_is_bullish': orb_is_bullish,
        'ny_is_bullish': ny_is_bullish,
        'high_broken': high_broken,
        'low_broken': low_broken,
        'first_break_direction': first_break_direction,
        
        # Timing
        'high_break_minutes': high_break_minutes,
        'low_break_minutes': low_break_minutes,
        
        # High breakout retests
        'high_retest_orb_high': high_retest_orb_high,
        'high_retest_midpoint': high_retest_midpoint,
        'high_retest_orb_low': high_retest_orb_low,
        'high_orb_high_continues': high_orb_high_continues,
        'high_midpoint_continues': high_midpoint_continues,
        'high_orb_low_continues': high_orb_low_continues,
        'high_orb_high_retest_minutes': high_orb_high_retest_minutes,
        'high_midpoint_retest_minutes': high_midpoint_retest_minutes,
        'high_orb_low_retest_minutes': high_orb_low_retest_minutes,
        'high_orb_high_mfe': high_orb_high_mfe,
        'high_orb_high_mae': high_orb_high_mae,
        'high_midpoint_mfe': high_midpoint_mfe,
        'high_midpoint_mae': high_midpoint_mae,
        'high_orb_low_mfe': high_orb_low_mfe,
        'high_orb_low_mae': high_orb_low_mae,
        
        # Low breakout retests
        'low_retest_orb_low': low_retest_orb_low,
        'low_retest_midpoint': low_retest_midpoint,
        'low_retest_orb_high': low_retest_orb_high,
        'low_orb_low_continues': low_orb_low_continues,
        'low_midpoint_continues': low_midpoint_continues,
        'low_orb_high_continues': low_orb_high_continues,
        'low_orb_low_retest_minutes': low_orb_low_retest_minutes,
        'low_midpoint_retest_minutes': low_midpoint_retest_minutes,
        'low_orb_high_retest_minutes': low_orb_high_retest_minutes,
        'low_orb_low_mfe': low_orb_low_mfe,
        'low_orb_low_mae': low_orb_low_mae,
        'low_midpoint_mfe': low_midpoint_mfe,
        'low_midpoint_mae': low_midpoint_mae,
        'low_orb_high_mfe': low_orb_high_mfe,
        'low_orb_high_mae': low_orb_high_mae,
    })

retest_df = pd.DataFrame(retest_data)

print(f"\n✓ Calculated retest data for {len(retest_df)} days")

# ============================================================
# MERGE WITH YOUR EXISTING DATASET
# ============================================================

input_dir = r"C:\Users\tammm\NQ and ES analysis"
existing_df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset.csv')
existing_df['date'] = pd.to_datetime(existing_df['date'])
retest_df['date'] = pd.to_datetime(retest_df['date'])

# Merge
merged_df = existing_df.merge(retest_df, on='date', how='left')

print(f"\n✓ Merged with existing dataset")
print(f"  Original columns: {len(existing_df.columns)}")
print(f"  New columns added: {len(retest_df.columns) - 1}")  # -1 for date
print(f"  Total columns: {len(merged_df.columns)}")

# Save
merged_df.to_csv(f'{input_dir}\\ultimate_comprehensive_dataset_with_retests.csv', index=False)
print(f"\n✓ Saved: ultimate_comprehensive_dataset_with_retests.csv")

# ============================================================
# NOW RUN ANALYSIS ON THE MERGED DATASET
# ============================================================

print("\n" + "="*100)
print("VALIDATION - Checking Against Your Feature Analysis")
print("="*100)

# Filter to rows with ORB data
orb_df = merged_df[merged_df['orb_935_direction'].notna()].copy()

print(f"\nRows with ORB data: {len(orb_df)}")

# Validate ORB direction
bullish_orb = orb_df[orb_df['orb_935_direction'] == 'bullish']
bearish_orb = orb_df[orb_df['orb_935_direction'] == 'bearish']

bull_orb_bull_ny = (bullish_orb['ny_direction'] == 'bullish').sum()
bear_orb_bear_ny = (bearish_orb['ny_direction'] == 'bearish').sum()

print(f"\nORB_935_DIRECTION validation:")
print(f"  Bullish ORB → Bullish NY: {bull_orb_bull_ny}/{len(bullish_orb)} = {bull_orb_bull_ny/len(bullish_orb)*100:.1f}%")
print(f"  Expected from feature analysis: 64.0%")
print(f"  Match: {'✓' if abs(bull_orb_bull_ny/len(bullish_orb)*100 - 64.0) < 1.0 else '✗ MISMATCH'}")

# Validate first break
high_first = orb_df[orb_df['orb_935_first_break'] == 'high']
low_first = orb_df[orb_df['orb_935_first_break'] == 'low']

high_first_bull_ny = (high_first['ny_direction'] == 'bullish').sum()
low_first_bear_ny = (low_first['ny_direction'] == 'bearish').sum()

print(f"\nORB_935_FIRST_BREAK validation:")
print(f"  HIGH first → Bullish NY: {high_first_bull_ny}/{len(high_first)} = {high_first_bull_ny/len(high_first)*100:.1f}%")
print(f"  Expected from feature analysis: 65.5%")
print(f"  Match: {'✓' if abs(high_first_bull_ny/len(high_first)*100 - 65.5) < 1.0 else '✗ MISMATCH'}")

print("\n" + "="*100)
print("If the validation passes, you now have:")
print("  1. All your original features")
print("  2. Retest data for the 5-minute ORB")
print("  3. MFE/MAE stats")
print("  4. Timing data")
print("  5. Continuation rates")
print("="*100)

5-MINUTE ORB (9:30-9:35) - COMPREHENSIVE RETEST ANALYSIS
Analyzing 3785 trading days
Matching your existing orb_935_direction and orb_935_first_break features

  Processing day 500...
  Processing day 1000...
  Processing day 1500...
  Processing day 2000...
  Processing day 2500...
  Processing day 3000...
  Processing day 3500...

✓ Calculated retest data for 3129 days

✓ Merged with existing dataset
  Original columns: 56
  New columns added: 42
  Total columns: 98

✓ Saved: ultimate_comprehensive_dataset_with_retests.csv

VALIDATION - Checking Against Your Feature Analysis

Rows with ORB data: 3125

ORB_935_DIRECTION validation:
  Bullish ORB → Bullish NY: 1007/1573 = 64.0%
  Expected from feature analysis: 64.0%
  Match: ✓

ORB_935_FIRST_BREAK validation:
  HIGH first → Bullish NY: 1076/1643 = 65.5%
  Expected from feature analysis: 65.5%
  Match: ✓

If the validation passes, you now have:
  1. All your original features
  2. Retest data for the 5-minute ORB
  3. MFE/MAE stats
  4

In [2]:
import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"
df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset_with_retests.csv')

print("="*100)
print("5-MINUTE ORB (9:30-9:35) - COMPREHENSIVE RETEST ANALYSIS")
print("="*100)
print(f"Analyzing {len(df)} trading days with complete retest data\n")

# Filter to rows with ORB data
orb_df = df[df['orb_935_direction'].notna()].copy()

print(f"Valid ORB days: {len(orb_df)}")

# ============================================================
# SECTION 1: DIRECTIONAL BIAS - ALL COMBINATIONS
# ============================================================

print("\n" + "="*100)
print("SECTION 1: DIRECTIONAL BIAS - ALL COMBINATIONS")
print("="*100)

combinations = [
    ('Bullish ORB + HIGH breaks first', 
     (orb_df['orb_935_direction'] == 'bullish') & (orb_df['orb_935_first_break'] == 'high')),
    ('Bullish ORB + LOW breaks first', 
     (orb_df['orb_935_direction'] == 'bullish') & (orb_df['orb_935_first_break'] == 'low')),
    ('Bearish ORB + HIGH breaks first', 
     (orb_df['orb_935_direction'] == 'bearish') & (orb_df['orb_935_first_break'] == 'high')),
    ('Bearish ORB + LOW breaks first', 
     (orb_df['orb_935_direction'] == 'bearish') & (orb_df['orb_935_first_break'] == 'low')),
]

print(f"\n{'Setup':<40} {'Sample':<10} {'Bull NY':<12} {'Bear NY':<12} {'Win%':<10} {'Edge':<10}")
print("-"*100)

for name, condition in combinations:
    subset = orb_df[condition]
    if len(subset) > 0:
        bull_ny = (subset['ny_direction'] == 'bullish').sum()
        bear_ny = len(subset) - bull_ny
        
        # Determine if bullish or bearish trade
        if 'Bullish' in name and 'HIGH' in name:
            win_rate = bull_ny / len(subset) * 100
            edge = win_rate - 54.6  # vs bullish baseline
        elif 'Bearish' in name and 'LOW' in name:
            win_rate = bear_ny / len(subset) * 100
            edge = win_rate - 45.4  # vs bearish baseline
        elif 'Bullish' in name and 'LOW' in name:
            win_rate = bull_ny / len(subset) * 100
            edge = win_rate - 54.6
        else:  # Bearish + HIGH
            win_rate = bear_ny / len(subset) * 100
            edge = win_rate - 45.4
        
        print(f"{name:<40} {len(subset):<10} {bull_ny:<12} {bear_ny:<12} {win_rate:<10.1f} {edge:>+9.1f}%")

# ============================================================
# SECTION 2: BREAKOUT TIMING ANALYSIS
# ============================================================

print("\n" + "="*100)
print("SECTION 2: BREAKOUT TIMING ANALYSIS")
print("="*100)

high_breaks = orb_df[orb_df['high_broken'] == True]
low_breaks = orb_df[orb_df['low_broken'] == True]

print(f"\nHIGH Breakout Timing (n={len(high_breaks)}):")
if len(high_breaks) > 0:
    high_times = orb_df[orb_df['high_break_minutes'].notna()]['high_break_minutes']
    print(f"  Mean:   {high_times.mean():.1f} minutes after ORB")
    print(f"  Median: {high_times.median():.1f} minutes after ORB")
    print(f"  25th %: {high_times.quantile(0.25):.1f} minutes")
    print(f"  75th %: {high_times.quantile(0.75):.1f} minutes")
    
    # 5-minute blocks
    print(f"\n  Distribution in 5-minute blocks:")
    for i in range(0, 36, 5):
        if i < 30:
            count = ((high_times >= i) & (high_times < i+5)).sum()
            pct = count / len(high_times) * 100
            time_str = f"{i}-{i+5} min"
        else:
            count = (high_times >= 30).sum()
            pct = count / len(high_times) * 100
            time_str = f"30+ min"
        print(f"    {time_str:<12}: {count:>4} ({pct:>5.1f}%)")
        if i >= 30:
            break

print(f"\nLOW Breakout Timing (n={len(low_breaks)}):")
if len(low_breaks) > 0:
    low_times = orb_df[orb_df['low_break_minutes'].notna()]['low_break_minutes']
    print(f"  Mean:   {low_times.mean():.1f} minutes after ORB")
    print(f"  Median: {low_times.median():.1f} minutes after ORB")
    print(f"  25th %: {low_times.quantile(0.25):.1f} minutes")
    print(f"  75th %: {low_times.quantile(0.75):.1f} minutes")
    
    # 5-minute blocks
    print(f"\n  Distribution in 5-minute blocks:")
    for i in range(0, 36, 5):
        if i < 30:
            count = ((low_times >= i) & (low_times < i+5)).sum()
            pct = count / len(low_times) * 100
            time_str = f"{i}-{i+5} min"
        else:
            count = (low_times >= 30).sum()
            pct = count / len(low_times) * 100
            time_str = f"30+ min"
        print(f"    {time_str:<12}: {count:>4} ({pct:>5.1f}%)")
        if i >= 30:
            break

# ============================================================
# SECTION 3: AFTER HIGH BREAKOUT - DETAILED RETEST ANALYSIS
# ============================================================

print("\n" + "="*100)
print("SECTION 3: AFTER HIGH BREAKOUT - DETAILED RETEST ANALYSIS")
print("="*100)

def analyze_retest_level(df, retest_col, continuation_col, retest_time_col, mfe_col, mae_col, level_name):
    retests = df[df[retest_col] == True]
    
    if len(retests) == 0:
        print(f"\n{level_name}: No retests found")
        return
    
    continues = retests[retests[continuation_col] == True]
    reverses = retests[retests[continuation_col] == False]
    undetermined = retests[retests[continuation_col].isna()]
    
    print(f"\n{level_name}")
    print(f"  Occurs: {len(retests)} times ({len(retests)/len(df)*100:.1f}% of breakouts)")
    print(f"  → Continuation: {len(continues)} ({len(continues)/len(retests)*100:.1f}%)")
    print(f"  → Reversal:     {len(reverses)} ({len(reverses)/len(retests)*100:.1f}%)")
    if len(undetermined) > 0:
        print(f"  → Undetermined: {len(undetermined)} ({len(undetermined)/len(retests)*100:.1f}%)")
    
    # Timing statistics
    retest_times = retests[retests[retest_time_col].notna()][retest_time_col]
    if len(retest_times) > 0:
        print(f"\n  Timing Statistics:")
        print(f"    Mean:   {retest_times.mean():.1f} minutes after ORB")
        print(f"    Median: {retest_times.median():.1f} minutes after ORB")
        print(f"    25th %: {retest_times.quantile(0.25):.1f} minutes")
        print(f"    75th %: {retest_times.quantile(0.75):.1f} minutes")
        
        # 5-minute blocks
        print(f"\n    Distribution in 5-minute blocks:")
        for i in range(0, 36, 5):
            if i < 30:
                count = ((retest_times >= i) & (retest_times < i+5)).sum()
                pct = count / len(retest_times) * 100 if len(retest_times) > 0 else 0
                time_str = f"{i}-{i+5} min"
            else:
                count = (retest_times >= 30).sum()
                pct = count / len(retest_times) * 100 if len(retest_times) > 0 else 0
                time_str = f"30+ min"
            print(f"      {time_str:<12}: {count:>4} ({pct:>5.1f}%)")
            if i >= 30:
                break
    
    # MFE/MAE statistics
    mfe_data = retests[retests[mfe_col].notna()][mfe_col]
    mae_data = retests[retests[mae_col].notna()][mae_col]
    
    if len(mfe_data) > 0:
        print(f"\n  Risk/Reward from Retest:")
        print(f"    MFE (Profit Potential):")
        print(f"      Mean:   {mfe_data.mean():.3f}%")
        print(f"      Median: {mfe_data.median():.3f}%")
        print(f"      25th %: {mfe_data.quantile(0.25):.3f}%")
        print(f"      75th %: {mfe_data.quantile(0.75):.3f}%")
        
        print(f"    MAE (Risk Exposure):")
        print(f"      Mean:   {mae_data.mean():.3f}%")
        print(f"      Median: {mae_data.median():.3f}%")
        print(f"      25th %: {mae_data.quantile(0.25):.3f}%")
        print(f"      75th %: {mae_data.quantile(0.75):.3f}%")
        
        if mae_data.median() > 0:
            print(f"    Median R:R Ratio: {mfe_data.median()/mae_data.median():.2f}:1")
    
    # When combined with bullish ORB
    bullish_retests = retests[retests['orb_935_direction'] == 'bullish']
    if len(bullish_retests) > 0:
        bullish_continues = bullish_retests[bullish_retests[continuation_col] == True]
        print(f"\n  When ORB is BULLISH (n={len(bullish_retests)}):")
        print(f"    Continuation: {len(bullish_continues)} ({len(bullish_continues)/len(bullish_retests)*100:.1f}%) 🎯")

# Analyze each retest level for HIGH breakout
analyze_retest_level(
    high_breaks, 
    'high_retest_orb_high', 
    'high_orb_high_continues',
    'high_orb_high_retest_minutes',
    'high_orb_high_mfe',
    'high_orb_high_mae',
    "1️⃣  RETEST OF ORB HIGH (shallow pullback to breakout level)"
)

analyze_retest_level(
    high_breaks,
    'high_retest_midpoint',
    'high_midpoint_continues',
    'high_midpoint_retest_minutes',
    'high_midpoint_mfe',
    'high_midpoint_mae',
    "2️⃣  RETEST OF MIDPOINT (moderate pullback)"
)

analyze_retest_level(
    high_breaks,
    'high_retest_orb_low',
    'high_orb_low_continues',
    'high_orb_low_retest_minutes',
    'high_orb_low_mfe',
    'high_orb_low_mae',
    "3️⃣  RETEST OF ORB LOW (deep pullback - full retracement)"
)

# ============================================================
# SECTION 4: AFTER LOW BREAKOUT - DETAILED RETEST ANALYSIS
# ============================================================

print("\n" + "="*100)
print("SECTION 4: AFTER LOW BREAKOUT - DETAILED RETEST ANALYSIS")
print("="*100)

# Analyze each retest level for LOW breakout
analyze_retest_level(
    low_breaks,
    'low_retest_orb_low',
    'low_orb_low_continues',
    'low_orb_low_retest_minutes',
    'low_orb_low_mfe',
    'low_orb_low_mae',
    "1️⃣  RETEST OF ORB LOW (shallow pullback to breakout level)"
)

analyze_retest_level(
    low_breaks,
    'low_retest_midpoint',
    'low_midpoint_continues',
    'low_midpoint_retest_minutes',
    'low_midpoint_mfe',
    'low_midpoint_mae',
    "2️⃣  RETEST OF MIDPOINT (moderate pullback)"
)

analyze_retest_level(
    low_breaks,
    'low_retest_orb_high',
    'low_orb_high_continues',
    'low_orb_high_retest_minutes',
    'low_orb_high_mfe',
    'low_orb_high_mae',
    "3️⃣  RETEST OF ORB HIGH (deep pullback - full retracement)"
)

# ============================================================
# SECTION 5: KEY INSIGHTS
# ============================================================

print("\n" + "="*100)
print("SECTION 5: KEY INSIGHTS & STRATEGY RECOMMENDATIONS")
print("="*100)

print("\n🎯 BEST SETUPS (Highest Win Rate):")
print("-"*100)

# Calculate best setups
setups = []
for name, condition in combinations:
    subset = orb_df[condition]
    if len(subset) > 0:
        bull_ny = (subset['ny_direction'] == 'bullish').sum()
        
        if 'Bullish' in name and 'HIGH' in name:
            win_rate = bull_ny / len(subset) * 100
            edge = win_rate - 54.6
            direction = 'LONG'
        elif 'Bearish' in name and 'LOW' in name:
            win_rate = (len(subset) - bull_ny) / len(subset) * 100
            edge = win_rate - 45.4
            direction = 'SHORT'
        else:
            continue
        
        setups.append({
            'name': name,
            'direction': direction,
            'win_rate': win_rate,
            'edge': edge,
            'sample': len(subset)
        })

setups_df = pd.DataFrame(setups).sort_values('edge', ascending=False)
for _, row in setups_df.iterrows():
    print(f"{row['direction']:<6} {row['name']:<40} Win: {row['win_rate']:.1f}% | Edge: {row['edge']:>+6.1f}% | n={row['sample']}")

print("\n" + "="*100)
print("ANALYSIS COMPLETE")
print("="*100)
print(f"\n✓ Full analysis exported to dataset with {len(orb_df.columns)} features")

5-MINUTE ORB (9:30-9:35) - COMPREHENSIVE RETEST ANALYSIS
Analyzing 3126 trading days with complete retest data

Valid ORB days: 3125

SECTION 1: DIRECTIONAL BIAS - ALL COMBINATIONS

Setup                                    Sample     Bull NY      Bear NY      Win%       Edge      
----------------------------------------------------------------------------------------------------
Bullish ORB + HIGH breaks first          1235       828          407          67.0           +12.4%
Bullish ORB + LOW breaks first           338        179          159          53.0            -1.6%
Bearish ORB + HIGH breaks first          408        248          160          39.2            -6.2%
Bearish ORB + LOW breaks first           1144       453          691          60.4           +15.0%

SECTION 2: BREAKOUT TIMING ANALYSIS

HIGH Breakout Timing (n=2653):
  Mean:   30.8 minutes after ORB
  Median: 5.0 minutes after ORB
  25th %: 1.0 minutes
  75th %: 23.0 minutes

  Distribution in 5-minute blocks:
  

In [3]:
import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"
df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset_with_retests.csv')

print("="*100)
print("9:35 ORB → NY SESSION DIRECTION PREDICTION ANALYSIS")
print("="*100)
print(f"Analyzing {len(df)} trading days\n")

# Filter to rows with ORB data
orb_df = df[df['orb_935_direction'].notna()].copy()

print(f"Valid ORB days: {len(orb_df)}")
print(f"Date range: {orb_df['date'].min()} to {orb_df['date'].max()}")

# ============================================================
# BASELINE - NY SESSION DIRECTION (NO ORB FILTER)
# ============================================================

print("\n" + "="*100)
print("BASELINE - NY SESSION DIRECTION (9:30 AM - 4:00 PM)")
print("="*100)

bullish_ny = (orb_df['ny_direction'] == 'bullish').sum()
bearish_ny = len(orb_df) - bullish_ny

print(f"\nOverall NY Session Direction:")
print(f"  Bullish: {bullish_ny} ({bullish_ny/len(orb_df)*100:.1f}%)")
print(f"  Bearish: {bearish_ny} ({bearish_ny/len(orb_df)*100:.1f}%)")
print(f"\n  Baseline win rate for LONG:  {bullish_ny/len(orb_df)*100:.1f}%")
print(f"  Baseline win rate for SHORT: {bearish_ny/len(orb_df)*100:.1f}%")

# ============================================================
# SECTION 1: ORB DIRECTION ONLY (9:30-9:35)
# ============================================================

print("\n" + "="*100)
print("SECTION 1: ORB DIRECTION PREDICTION (9:30-9:35 candle only)")
print("="*100)

bullish_orb = orb_df[orb_df['orb_935_direction'] == 'bullish']
bearish_orb = orb_df[orb_df['orb_935_direction'] == 'bearish']

print(f"\n{'ORB Direction':<20} {'Sample':<10} {'→ Bull NY':<15} {'→ Bear NY':<15} {'Win%':<10} {'Edge':<10}")
print("-"*100)

# Bullish ORB
bull_orb_to_bull_ny = (bullish_orb['ny_direction'] == 'bullish').sum()
bull_orb_to_bear_ny = len(bullish_orb) - bull_orb_to_bull_ny
bull_win_rate = bull_orb_to_bull_ny / len(bullish_orb) * 100
bull_edge = bull_win_rate - (bullish_ny/len(orb_df)*100)

print(f"{'Bullish ORB':<20} {len(bullish_orb):<10} {bull_orb_to_bull_ny:<15} {bull_orb_to_bear_ny:<15} {bull_win_rate:<10.1f} {bull_edge:>+9.1f}%")

# Bearish ORB
bear_orb_to_bull_ny = (bearish_orb['ny_direction'] == 'bullish').sum()
bear_orb_to_bear_ny = len(bearish_orb) - bear_orb_to_bull_ny
bear_win_rate = bear_orb_to_bear_ny / len(bearish_orb) * 100
bear_edge = bear_win_rate - (bearish_ny/len(orb_df)*100)

print(f"{'Bearish ORB':<20} {len(bearish_orb):<10} {bear_orb_to_bull_ny:<15} {bear_orb_to_bear_ny:<15} {bear_win_rate:<10.1f} {bear_edge:>+9.1f}%")

print(f"\n📊 INTERPRETATION:")
print(f"  • A BULLISH ORB predicts bullish NY session {bull_win_rate:.1f}% of the time (edge: {bull_edge:+.1f}%)")
print(f"  • A BEARISH ORB predicts bearish NY session {bear_win_rate:.1f}% of the time (edge: {bear_edge:+.1f}%)")

# ============================================================
# SECTION 2: ORB DIRECTION + FIRST BREAKOUT
# ============================================================

print("\n" + "="*100)
print("SECTION 2: ORB DIRECTION + FIRST BREAKOUT COMBINATION")
print("="*100)
print("Does knowing WHICH level breaks first improve prediction?")

print(f"\n{'Setup':<35} {'Sample':<10} {'→ Bull NY':<12} {'→ Bear NY':<12} {'Win%':<10} {'Edge':<10}")
print("-"*100)

# Define all combinations
combinations = [
    ('Bullish ORB → HIGH breaks first', 
     (orb_df['orb_935_direction'] == 'bullish') & (orb_df['orb_935_first_break'] == 'high'),
     'bullish'),
    ('Bullish ORB → LOW breaks first', 
     (orb_df['orb_935_direction'] == 'bullish') & (orb_df['orb_935_first_break'] == 'low'),
     'bullish'),
    ('Bearish ORB → HIGH breaks first', 
     (orb_df['orb_935_direction'] == 'bearish') & (orb_df['orb_935_first_break'] == 'high'),
     'bearish'),
    ('Bearish ORB → LOW breaks first', 
     (orb_df['orb_935_direction'] == 'bearish') & (orb_df['orb_935_first_break'] == 'low'),
     'bearish'),
]

results = []

for name, condition, expected_direction in combinations:
    subset = orb_df[condition]
    if len(subset) > 0:
        to_bull_ny = (subset['ny_direction'] == 'bullish').sum()
        to_bear_ny = len(subset) - to_bull_ny
        
        # Win rate = % that matches expected direction
        if expected_direction == 'bullish':
            win_rate = to_bull_ny / len(subset) * 100
            baseline = bullish_ny/len(orb_df)*100
        else:
            win_rate = to_bear_ny / len(subset) * 100
            baseline = bearish_ny/len(orb_df)*100
        
        edge = win_rate - baseline
        
        print(f"{name:<35} {len(subset):<10} {to_bull_ny:<12} {to_bear_ny:<12} {win_rate:<10.1f} {edge:>+9.1f}%")
        
        results.append({
            'setup': name,
            'sample': len(subset),
            'win_rate': win_rate,
            'edge': edge,
            'expected_direction': expected_direction
        })

# ============================================================
# SECTION 3: ORB RANGE SIZE ANALYSIS
# ============================================================

print("\n" + "="*100)
print("SECTION 3: ORB RANGE SIZE IMPACT")
print("="*100)
print("Does a larger/smaller ORB range affect prediction accuracy?")

# Calculate quartiles for ORB range
orb_df['orb_range_quartile'] = pd.qcut(orb_df['orb_935_range_pct'], 
                                         q=4, 
                                         labels=['Q1 (Smallest)', 'Q2', 'Q3', 'Q4 (Largest)'])

print(f"\n{'ORB Range':<20} {'Sample':<10} {'Mean Range %':<15} {'Bull ORB→Bull NY':<20} {'Bear ORB→Bear NY':<20}")
print("-"*100)

for quartile in ['Q1 (Smallest)', 'Q2', 'Q3', 'Q4 (Largest)']:
    quartile_data = orb_df[orb_df['orb_range_quartile'] == quartile]
    
    # Bullish ORB in this quartile
    bull_orb_q = quartile_data[quartile_data['orb_935_direction'] == 'bullish']
    bull_success = (bull_orb_q['ny_direction'] == 'bullish').sum()
    bull_rate = (bull_success / len(bull_orb_q) * 100) if len(bull_orb_q) > 0 else 0
    
    # Bearish ORB in this quartile
    bear_orb_q = quartile_data[quartile_data['orb_935_direction'] == 'bearish']
    bear_success = (bear_orb_q['ny_direction'] == 'bearish').sum()
    bear_rate = (bear_success / len(bear_orb_q) * 100) if len(bear_orb_q) > 0 else 0
    
    mean_range = quartile_data['orb_935_range_pct'].mean()
    
    print(f"{quartile:<20} {len(quartile_data):<10} {mean_range:<15.3f} {bull_rate:<20.1f}% {bear_rate:<20.1f}%")

# ============================================================
# SECTION 4: TIMING - WHEN DOES FIRST BREAK OCCUR?
# ============================================================

print("\n" + "="*100)
print("SECTION 4: BREAKOUT TIMING IMPACT")
print("="*100)
print("Does WHEN the first breakout occurs affect NY session prediction?")

# Create timing bins
orb_df['first_break_minutes'] = orb_df.apply(
    lambda row: row['high_break_minutes'] if row['orb_935_first_break'] == 'high' 
    else row['low_break_minutes'], axis=1
)

# Define timing bins
timing_bins = [
    ('0-5 min (immediate)', 0, 5),
    ('5-15 min (early)', 5, 15),
    ('15-30 min (mid)', 15, 30),
    ('30+ min (late)', 30, 1000),
]

print(f"\n{'Break Timing':<25} {'Sample':<10} {'Bull ORB→Bull NY':<20} {'Bear ORB→Bear NY':<20}")
print("-"*100)

for label, min_time, max_time in timing_bins:
    if max_time == 1000:
        timing_subset = orb_df[orb_df['first_break_minutes'] >= min_time]
    else:
        timing_subset = orb_df[(orb_df['first_break_minutes'] >= min_time) & 
                               (orb_df['first_break_minutes'] < max_time)]
    
    if len(timing_subset) > 0:
        # Bullish ORB
        bull_orb_timing = timing_subset[timing_subset['orb_935_direction'] == 'bullish']
        bull_success = (bull_orb_timing['ny_direction'] == 'bullish').sum()
        bull_rate = (bull_success / len(bull_orb_timing) * 100) if len(bull_orb_timing) > 0 else 0
        
        # Bearish ORB
        bear_orb_timing = timing_subset[timing_subset['orb_935_direction'] == 'bearish']
        bear_success = (bear_orb_timing['ny_direction'] == 'bearish').sum()
        bear_rate = (bear_success / len(bear_orb_timing) * 100) if len(bear_orb_timing) > 0 else 0
        
        print(f"{label:<25} {len(timing_subset):<10} {bull_rate:<20.1f}% {bear_rate:<20.1f}%")

# ============================================================
# SECTION 5: BEST PREDICTIVE COMBINATIONS
# ============================================================

print("\n" + "="*100)
print("SECTION 5: RANKED PREDICTIONS - BEST TO WORST")
print("="*100)

results_df = pd.DataFrame(results).sort_values('edge', ascending=False)

print(f"\n{'Rank':<6} {'Setup':<40} {'Win%':<10} {'Edge':<12} {'Sample':<10} {'Trade Direction':<15}")
print("-"*100)

for idx, row in results_df.iterrows():
    rank = results_df.index.get_loc(idx) + 1
    trade_dir = 'LONG' if row['expected_direction'] == 'bullish' else 'SHORT'
    print(f"{rank:<6} {row['setup']:<40} {row['win_rate']:<10.1f} {row['edge']:>+10.1f}% {row['sample']:<10} {trade_dir:<15}")

# ============================================================
# SECTION 6: KEY INSIGHTS & TRADING RULES
# ============================================================

print("\n" + "="*100)
print("SECTION 6: KEY INSIGHTS & TRADING RULES")
print("="*100)

best_long = results_df[results_df['expected_direction'] == 'bullish'].iloc[0]
best_short = results_df[results_df['expected_direction'] == 'bearish'].iloc[0]

print(f"\n🎯 BEST LONG SETUP:")
print(f"   {best_long['setup']}")
print(f"   Win Rate: {best_long['win_rate']:.1f}% | Edge: {best_long['edge']:+.1f}% | Sample: {best_long['sample']}")

print(f"\n🎯 BEST SHORT SETUP:")
print(f"   {best_short['setup']}")
print(f"   Win Rate: {best_short['win_rate']:.1f}% | Edge: {best_short['edge']:+.1f}% | Sample: {best_short['sample']}")

print(f"\n📊 SUMMARY:")
print(f"   • ORB direction alone gives +{bull_edge:.1f}% to +{bear_edge:.1f}% edge")
print(f"   • Adding first breakout direction improves edge to +{best_long['edge']:.1f}% (long) and +{best_short['edge']:.1f}% (short)")
print(f"   • Best improvement: {max(best_long['edge'] - bull_edge, best_short['edge'] - bear_edge):.1f}% additional edge from breakout filter")

print("\n✅ TRADING RULES:")
print("   1. Wait for 9:35 AM ORB to complete")
print("   2. Identify ORB direction (bullish/bearish based on close vs open)")
print("   3. Wait for first breakout (high or low)")
print("   4. Enter trade if setup matches high-edge combinations above")
print("   5. Target: Hold until 4:00 PM close for directional trade")

print("\n❌ AVOID:")
worst_long = results_df[results_df['expected_direction'] == 'bullish'].iloc[-1]
worst_short = results_df[results_df['expected_direction'] == 'bearish'].iloc[-1]

print(f"   • {worst_long['setup']}: {worst_long['win_rate']:.1f}% (edge: {worst_long['edge']:+.1f}%)")
print(f"   • {worst_short['setup']}: {worst_short['win_rate']:.1f}% (edge: {worst_short['edge']:+.1f}%)")

print("\n" + "="*100)
print("ANALYSIS COMPLETE")
print("="*100)

9:35 ORB → NY SESSION DIRECTION PREDICTION ANALYSIS
Analyzing 3126 trading days

Valid ORB days: 3125
Date range: 2013-01-03 to 2025-03-14

BASELINE - NY SESSION DIRECTION (9:30 AM - 4:00 PM)

Overall NY Session Direction:
  Bullish: 1708 (54.7%)
  Bearish: 1417 (45.3%)

  Baseline win rate for LONG:  54.7%
  Baseline win rate for SHORT: 45.3%

SECTION 1: ORB DIRECTION PREDICTION (9:30-9:35 candle only)

ORB Direction        Sample     → Bull NY       → Bear NY       Win%       Edge      
----------------------------------------------------------------------------------------------------
Bullish ORB          1573       1007            566             64.0            +9.4%
Bearish ORB          1552       701             851             54.8            +9.5%

📊 INTERPRETATION:
  • A BULLISH ORB predicts bullish NY session 64.0% of the time (edge: +9.4%)
  • A BEARISH ORB predicts bearish NY session 54.8% of the time (edge: +9.5%)

SECTION 2: ORB DIRECTION + FIRST BREAKOUT COMBINATION
Doe

In [ ]:
"""
5-Minute ORB Strategy Backtester + Monte Carlo Simulation
NQ Futures — 1-minute OHLCV data

ENTRY TYPES:
    breakout    — enter at ORB high/low on the first 5-min bar that closes beyond it
                  stop: midpoint or 25pt fixed | target: fixed R:R (1:1, 1.5:1, 2:1)

    retest      — enter at ORB high/low on first clean retest after breakout
                  invalidated if price hits midpoint first
                  stop: midpoint or 25pt fixed | target: fixed R:R (1:1, 1.5:1, 2:1)

    1min_break  — enter on open of bar AFTER first 1-min close beyond ORB high/low
                  stop: ORB midpoint
                  target: ORB high/low + (ORB range x extension%)
                  extensions tested: 25%, 50%, 75%, 100%

TOTAL VARIANTS: 16
    breakout   x 2 stops x 3 R:Rs  =  6
    retest     x 2 stops x 3 R:Rs  =  6
    1min_break x 1 stop  x 4 exts  =  4

EXPECTED CSV FORMAT:
    Date, Time, Open, High, Low, Close, Volume
    2010-06-06, 20:00:00, 1826.25, 1826.50, 1826.00, 1826.50, 6

OUTPUTS:
    orb_results_summary.csv   -- performance matrix, all 16 variants
    orb_trades_full.csv       -- every individual trade
    orb_montecarlo.csv        -- Monte Carlo summary
    orb_equity_curves.png     -- top 6 equity curves
    orb_montecarlo.png        -- MC distribution charts
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import warnings
import os

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG -- edit paths and parameters here
# ─────────────────────────────────────────────────────────────────────────────

INPUT_FILE = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
OUTPUT_DIR = r"C:\Users\tammm\NQ and ES analysis\orb_results"

SESSION_START  = "09:30"
ORB_END        = "09:35"   # ORB = first 5-min candle 09:30-09:35
SESSION_END    = "16:00"
NO_ENTRY_AFTER = "15:00"   # no new entries after this time

# Existing entry variants (breakout + retest)
STOP_TYPES = ["midpoint", "25pt"]   # 50pt removed
RR_TARGETS = [1.0, 1.5, 2.0]

# 1-min breakout entry variants (extension as fraction of ORB range)
EXTENSIONS = [0.25, 0.50, 0.75, 1.00]

POINT_VALUE       = 20       # NQ: $20 per point per contract
CONTRACTS         = 1

MC_SIMULATIONS    = 10_000
MC_TRADE_COUNT    = 200      # trades per simulation path
MC_RUIN_THRESHOLD = -15_000  # drawdown level that counts as ruin ($)


# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_data(filepath):
    print(f"\n Loading data from: {filepath}")
    df = pd.read_csv(filepath)
    print(f"   Raw shape:  {df.shape}")
    print(f"   Columns:    {list(df.columns)}")

    df["datetime"] = pd.to_datetime(
        df["Date"].astype(str) + " " + df["Time"].astype(str)
    )
    df = df.rename(columns={
        "Open": "open", "High": "high",
        "Low":  "low",  "Close": "close", "Volume": "volume"
    })
    for col in ["open", "high", "low", "close"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["open", "high", "low", "close"])
    df = df.sort_values("datetime").reset_index(drop=True)
    df["date"]     = df["datetime"].dt.date
    df["time_str"] = df["datetime"].dt.strftime("%H:%M")

    print(f"   Clean shape: {df.shape}")
    print(f"   Date range:  {df['date'].min()} to {df['date'].max()}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# ORB IDENTIFICATION
# ─────────────────────────────────────────────────────────────────────────────

def build_orb_table(df):
    """Aggregate 1-min bars into 5-min ORB candle (09:30-09:35) per day."""
    session = df[
        (df["time_str"] >= SESSION_START) &
        (df["time_str"] <  ORB_END)
    ].copy()

    orb = session.groupby("date").agg(
        orb_open  = ("open",  "first"),
        orb_high  = ("high",  "max"),
        orb_low   = ("low",   "min"),
        orb_close = ("close", "last"),
    ).reset_index()

    orb["orb_mid"]       = (orb["orb_high"] + orb["orb_low"]) / 2
    orb["orb_range"]     = orb["orb_high"] - orb["orb_low"]
    orb["orb_direction"] = np.where(
        orb["orb_close"] >= orb["orb_open"], "bullish", "bearish"
    )

    print(f"\n ORB Table: {len(orb)} days | "
          f"Bull: {(orb['orb_direction']=='bullish').sum()} | "
          f"Bear: {(orb['orb_direction']=='bearish').sum()} | "
          f"Mean range: {orb['orb_range'].mean():.1f} pts")
    return orb


# ─────────────────────────────────────────────────────────────────────────────
# BREAKOUT DETECTION
# ─────────────────────────────────────────────────────────────────────────────

def detect_breakouts(df, orb):
    """Find which ORB level broke first after 09:35 on each day."""
    post_orb = df[df["time_str"] >= ORB_END].copy()
    results  = []

    for _, row in orb.iterrows():
        day_data = post_orb[post_orb["date"] == row["date"]]
        if day_data.empty:
            continue

        hi_idx  = day_data[day_data["high"] > row["orb_high"]].index
        lo_idx  = day_data[day_data["low"]  < row["orb_low"]].index
        hi_time = day_data.loc[hi_idx[0], "datetime"] if len(hi_idx) else pd.NaT
        lo_time = day_data.loc[lo_idx[0], "datetime"] if len(lo_idx) else pd.NaT

        if pd.isna(hi_time) and pd.isna(lo_time):
            first_break = "none"
        elif pd.isna(lo_time):
            first_break = "high"
        elif pd.isna(hi_time):
            first_break = "low"
        else:
            first_break = "high" if hi_time <= lo_time else "low"

        results.append({
            **row.to_dict(),
            "high_break_time": hi_time,
            "low_break_time":  lo_time,
            "first_break":     first_break,
        })

    result_df = pd.DataFrame(results)
    result_df["valid_setup"] = (
        ((result_df["orb_direction"] == "bullish") & (result_df["first_break"] == "high")) |
        ((result_df["orb_direction"] == "bearish") & (result_df["first_break"] == "low"))
    )

    valid = result_df["valid_setup"].sum()
    print(f" Valid high-edge setups: {valid} / {len(result_df)} "
          f"({100*valid/len(result_df):.1f}%)")
    return result_df


# ─────────────────────────────────────────────────────────────────────────────
# TRADE SIMULATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def compute_stop_and_target(entry_price, direction, stop_type,
                             orb_high, orb_low, orb_mid, rr):
    if direction == "long":
        stop   = orb_mid if stop_type == "midpoint" else entry_price - 25
        risk   = entry_price - stop
        target = entry_price + risk * rr
    else:
        stop   = orb_mid if stop_type == "midpoint" else entry_price + 25
        risk   = stop - entry_price
        target = entry_price - risk * rr
    return stop, target, abs(risk)


def simulate_trade(day_data, entry_time, entry_price, stop, target, direction):
    """Walk forward bar-by-bar to find stop/target hit or session close."""
    bars = day_data[
        (day_data["datetime"] > entry_time) &
        (day_data["time_str"] <= SESSION_END)
    ]
    mfe = mae = 0.0

    for _, bar in bars.iterrows():
        if direction == "long":
            bar_mfe = bar["high"] - entry_price
            bar_mae = entry_price - bar["low"]
            if bar["low"] <= stop:
                return {"outcome": "loss", "exit_price": stop,
                        "exit_time": bar["datetime"],
                        "mfe": max(mfe, bar_mfe), "mae": max(mae, bar_mae)}
            if bar["high"] >= target:
                return {"outcome": "win", "exit_price": target,
                        "exit_time": bar["datetime"],
                        "mfe": max(mfe, bar_mfe), "mae": max(mae, bar_mae)}
        else:
            bar_mfe = entry_price - bar["low"]
            bar_mae = bar["high"] - entry_price
            if bar["high"] >= stop:
                return {"outcome": "loss", "exit_price": stop,
                        "exit_time": bar["datetime"],
                        "mfe": max(mfe, bar_mfe), "mae": max(mae, bar_mae)}
            if bar["low"] <= target:
                return {"outcome": "win", "exit_price": target,
                        "exit_time": bar["datetime"],
                        "mfe": max(mfe, bar_mfe), "mae": max(mae, bar_mae)}

        mfe = max(mfe, bar["high"] - entry_price if direction == "long"
                       else entry_price - bar["low"])
        mae = max(mae, entry_price - bar["low"] if direction == "long"
                       else bar["high"] - entry_price)

    if not bars.empty:
        last    = bars.iloc[-1]
        exit_px = last["close"]
        pnl     = (exit_px - entry_price) if direction == "long" \
                  else (entry_price - exit_px)
        return {"outcome": "win" if pnl > 0 else "loss",
                "exit_price": exit_px, "exit_time": last["datetime"],
                "mfe": mfe, "mae": mae}

    return {"outcome": "no_exit", "exit_price": entry_price,
            "exit_time": entry_time, "mfe": 0.0, "mae": 0.0}


def record_trade(base, entry_type, stop_type, rr, extension,
                 entry_price, stop, target, risk_pts,
                 result, direction, orb_range):
    pnl_pts = (result["exit_price"] - entry_price) if direction == "long" \
              else (entry_price - result["exit_price"])
    return {
        **base,
        "entry_type":  entry_type,
        "stop_type":   stop_type,
        "rr":          rr,
        "extension":   extension,
        "entry_price": entry_price,
        "stop":        stop,
        "target":      target,
        "risk_pts":    risk_pts,
        "outcome":     result["outcome"],
        "exit_price":  result["exit_price"],
        "exit_time":   result["exit_time"],
        "pnl_pts":     pnl_pts,
        "pnl_dollars": pnl_pts * POINT_VALUE * CONTRACTS,
        "mfe":         result["mfe"],
        "mae":         result["mae"],
        "orb_range":   orb_range,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MAIN BACKTEST LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_backtest(df, orb_df):
    n_variants = len(STOP_TYPES) * len(RR_TARGETS) * 2 + len(EXTENSIONS)
    trades     = []
    valid_days = orb_df[orb_df["valid_setup"]].copy()
    total      = len(valid_days)

    print(f"\n Running backtest on {total} valid setup days...")
    print(f"   2 entries x {len(STOP_TYPES)} stops x {len(RR_TARGETS)} R:Rs "
          f"+ 1min_break x {len(EXTENSIONS)} extensions = {n_variants} variants")

    for i, (_, setup) in enumerate(valid_days.iterrows()):
        if i % 200 == 0:
            print(f"   Day {i}/{total}...")

        date      = setup["date"]
        direction = "long" if setup["orb_direction"] == "bullish" else "short"
        orb_high  = setup["orb_high"]
        orb_low   = setup["orb_low"]
        orb_mid   = setup["orb_mid"]
        orb_range = setup["orb_range"]

        day_data = df[df["date"] == date].copy()
        day_reg  = day_data[
            (day_data["time_str"] >= SESSION_START) &
            (day_data["time_str"] <= SESSION_END)
        ]
        base = {"date": date, "direction": direction}

        # ── 1-MIN BREAKOUT ENTRY ──────────────────────────────────────────
        # First 1-min bar whose CLOSE is beyond the ORB level
        post_orb    = day_reg[day_reg["time_str"] >= ORB_END].copy()
        trigger_bar = None

        for _, bar in post_orb.iterrows():
            if bar["time_str"] > NO_ENTRY_AFTER:
                break
            if direction == "long"  and bar["close"] > orb_high:
                trigger_bar = bar
                break
            if direction == "short" and bar["close"] < orb_low:
                trigger_bar = bar
                break

        if trigger_bar is not None:
            next_bars = post_orb[post_orb["datetime"] > trigger_bar["datetime"]]
            if not next_bars.empty:
                entry_bar   = next_bars.iloc[0]
                entry_price = entry_bar["open"]
                entry_time  = entry_bar["datetime"]

                if entry_bar["time_str"] <= NO_ENTRY_AFTER:
                    stop = orb_mid
                    risk = abs(entry_price - stop)

                    if risk > 0:
                        for ext in EXTENSIONS:
                            ext_pts = orb_range * ext
                            target  = (orb_high + ext_pts) if direction == "long" \
                                      else (orb_low  - ext_pts)
                            result  = simulate_trade(
                                day_data, entry_time,
                                entry_price, stop, target, direction
                            )
                            trades.append(record_trade(
                                base, "1min_break", "midpoint", None, ext,
                                entry_price, stop, target, risk,
                                result, direction, orb_range
                            ))

        # ── 5-MIN BREAKOUT ENTRY ──────────────────────────────────────────
        break_time = setup["high_break_time"] if direction == "long" \
                     else setup["low_break_time"]
        if pd.isna(break_time):
            continue

        break_bar_rows = day_data[day_data["datetime"] >= break_time]
        if break_bar_rows.empty:
            continue

        breakout_bar         = break_bar_rows.iloc[0]
        breakout_entry_price = orb_high if direction == "long" else orb_low

        if breakout_bar["time_str"] <= NO_ENTRY_AFTER:
            for stop_type in STOP_TYPES:
                for rr in RR_TARGETS:
                    stop, target, risk_pts = compute_stop_and_target(
                        breakout_entry_price, direction, stop_type,
                        orb_high, orb_low, orb_mid, rr
                    )
                    if risk_pts <= 0:
                        continue
                    result = simulate_trade(
                        day_data, breakout_bar["datetime"],
                        breakout_entry_price, stop, target, direction
                    )
                    trades.append(record_trade(
                        base, "breakout", stop_type, rr, None,
                        breakout_entry_price, stop, target, risk_pts,
                        result, direction, orb_range
                    ))

        # ── RETEST ENTRY ──────────────────────────────────────────────────
        retest_level = orb_high if direction == "long" else orb_low
        post_break   = day_data[
            (day_data["datetime"] > breakout_bar["datetime"]) &
            (day_data["time_str"] <= NO_ENTRY_AFTER)
        ]

        retest_price = None
        retest_bar   = None
        invalidated  = False

        for _, bar in post_break.iterrows():
            if direction == "long"  and bar["low"]  <= orb_mid:
                invalidated = True; break
            if direction == "short" and bar["high"] >= orb_mid:
                invalidated = True; break
            if direction == "long":
                if bar["low"] <= retest_level and bar["close"] >= retest_level:
                    retest_price = retest_level
                    retest_bar   = bar
                    break
            else:
                if bar["high"] >= retest_level and bar["close"] <= retest_level:
                    retest_price = retest_level
                    retest_bar   = bar
                    break

        if not invalidated and retest_price is not None:
            for stop_type in STOP_TYPES:
                for rr in RR_TARGETS:
                    stop, target, risk_pts = compute_stop_and_target(
                        retest_price, direction, stop_type,
                        orb_high, orb_low, orb_mid, rr
                    )
                    if risk_pts <= 0:
                        continue
                    result = simulate_trade(
                        day_data, retest_bar["datetime"],
                        retest_price, stop, target, direction
                    )
                    trades.append(record_trade(
                        base, "retest", stop_type, rr, None,
                        retest_price, stop, target, risk_pts,
                        result, direction, orb_range
                    ))

    trades_df = pd.DataFrame(trades)
    print(f"\n Backtest complete: {len(trades_df):,} trade records")
    return trades_df


# ─────────────────────────────────────────────────────────────────────────────
# PERFORMANCE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

def summarise_results(trades_df):
    clean = trades_df[trades_df["outcome"].isin(["win", "loss"])].copy()

    def variant_label(row):
        if row["entry_type"] == "1min_break":
            return f"1min_break | midpoint | {int(row['extension']*100)}% ext"
        return f"{row['entry_type']} | {row['stop_type']} | {row['rr']}:1"

    clean["variant"] = clean.apply(variant_label, axis=1)
    rows = []

    for variant, g in clean.groupby("variant"):
        # CRITICAL: sort chronologically so drawdown is meaningful
        g = g.sort_values("date").reset_index(drop=True)

        n         = len(g)
        wins      = (g["outcome"] == "win").sum()
        win_rate  = wins / n
        avg_win   = g[g["outcome"]=="win"]["pnl_dollars"].mean()
        avg_loss  = g[g["outcome"]=="loss"]["pnl_dollars"].mean()
        total_pnl = g["pnl_dollars"].sum()
        avg_risk  = g["risk_pts"].mean()

        gross_profit = g[g["pnl_dollars"] > 0]["pnl_dollars"].sum()
        gross_loss   = abs(g[g["pnl_dollars"] < 0]["pnl_dollars"].sum())
        pf           = gross_profit / gross_loss if gross_loss > 0 else np.inf

        # Chronological drawdown
        cum    = g["pnl_dollars"].cumsum().values
        peak   = np.maximum.accumulate(cum)
        dd     = cum - peak
        max_dd = dd.min()

        # Worst consecutive loss streak
        max_streak = cur_streak = 0
        for o in g["outcome"].values:
            cur_streak = cur_streak + 1 if o == "loss" else 0
            max_streak = max(max_streak, cur_streak)

        expectancy = (win_rate * avg_win) + ((1 - win_rate) * avg_loss)

        sample = g.iloc[0]
        rows.append({
            "variant":             variant,
            "entry_type":          sample["entry_type"],
            "stop_type":           sample["stop_type"],
            "rr":                  sample["rr"],
            "extension":           sample["extension"],
            "n_trades":            n,
            "win_rate":            round(win_rate * 100, 1),
            "avg_win_$":           round(avg_win,  0),
            "avg_loss_$":          round(avg_loss, 0),
            "profit_factor":       round(pf, 2),
            "expectancy_$":        round(expectancy, 0),
            "total_pnl_$":         round(total_pnl, 0),
            "max_drawdown_$":      round(max_dd, 0),
            "max_single_loss_$":   round(g["pnl_dollars"].min(), 0),
            "max_consec_losses":   max_streak,
            "avg_risk_pts":        round(avg_risk, 1),
        })

    return pd.DataFrame(rows).sort_values("expectancy_$", ascending=False)


# ─────────────────────────────────────────────────────────────────────────────
# MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────

def run_monte_carlo(trades_df, variant_row):
    entry_type = variant_row["entry_type"]
    stop_type  = variant_row["stop_type"]
    rr         = variant_row["rr"]
    extension  = variant_row["extension"]

    mask  = trades_df["outcome"].isin(["win", "loss"])
    mask &= trades_df["entry_type"] == entry_type
    mask &= trades_df["stop_type"]  == stop_type

    if entry_type == "1min_break":
        mask &= trades_df["extension"] == extension
    else:
        mask &= trades_df["rr"] == rr

    pnl_arr = trades_df[mask]["pnl_dollars"].values
    if len(pnl_arr) < 10:
        return {}

    ending_equity = []
    max_drawdowns = []
    ruin_count    = 0

    np.random.seed(42)
    for _ in range(MC_SIMULATIONS):
        sample = np.random.choice(pnl_arr, size=MC_TRADE_COUNT, replace=True)
        cum    = np.cumsum(sample)
        peak   = np.maximum.accumulate(cum)
        dd     = cum - peak
        ending_equity.append(cum[-1])
        max_drawdowns.append(dd.min())
        if dd.min() <= MC_RUIN_THRESHOLD:
            ruin_count += 1

    return {
        "variant":            variant_row["variant"],
        "median_end_equity":  np.median(ending_equity),
        "p5_end_equity":      np.percentile(ending_equity, 5),
        "p95_end_equity":     np.percentile(ending_equity, 95),
        "median_max_dd":      np.median(max_drawdowns),
        "p5_max_dd":          np.percentile(max_drawdowns, 5),
        "prob_ruin_%":        round(100 * ruin_count / MC_SIMULATIONS, 1),
        "ending_equity_dist": ending_equity,
        "max_dd_dist":        max_drawdowns,
    }


# ─────────────────────────────────────────────────────────────────────────────
# CHARTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_equity_curves(trades_df, summary_df, output_dir):
    top  = summary_df.head(6)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor("#0d1117")
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, top.iterrows()):
        mask = (
            (trades_df["entry_type"] == row["entry_type"]) &
            (trades_df["stop_type"]  == row["stop_type"])  &
            (trades_df["outcome"].isin(["win", "loss"]))
        )
        if row["entry_type"] == "1min_break":
            mask &= trades_df["extension"] == row["extension"]
        else:
            mask &= trades_df["rr"] == row["rr"]

        g = trades_df[mask].copy()
        g["cum"] = g["pnl_dollars"].cumsum()

        ax.set_facecolor("#0d1117")
        ax.plot(g["cum"].values, color="#00ff9f", linewidth=1.2)
        ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
        ax.fill_between(range(len(g)), g["cum"].values, 0,
                        where=(g["cum"].values >= 0), alpha=0.12, color="#00ff9f")
        ax.fill_between(range(len(g)), g["cum"].values, 0,
                        where=(g["cum"].values <  0), alpha=0.12, color="#ff4455")

        ax.set_title(row["variant"], color="white", fontsize=8.5,
                     fontweight="bold", pad=4)
        ax.text(0.5, 0.96,
                f"WR: {row['win_rate']}%  PF: {row['profit_factor']}  "
                f"Total: ${row['total_pnl_$']:,.0f}  MaxDD: ${row['max_drawdown_$']:,.0f}",
                transform=ax.transAxes, ha="center", va="top",
                color="#aaa", fontsize=7)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax.tick_params(colors="#666", labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor("#333")

    fig.suptitle("ORB Strategy -- Top 6 Variants by Expectancy",
                 color="white", fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(output_dir, "orb_equity_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.close()
    print(f"   Saved: {path}")


def plot_monte_carlo(mc_results, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor("#0d1117")
    colors = ["#00ff9f", "#00cfff", "#ffcc00", "#ff7a45", "#cc88ff", "#ff4455"]

    for ax in axes:
        ax.set_facecolor("#0d1117")
        ax.tick_params(colors="#888", labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#333")

    for i, mc in enumerate(mc_results[:6]):
        c = colors[i % len(colors)]
        axes[0].hist(mc["ending_equity_dist"], bins=80, alpha=0.5,
                     color=c, label=mc["variant"], density=True)
        axes[1].hist(mc["max_dd_dist"], bins=80, alpha=0.5,
                     color=c, label=mc["variant"], density=True)

    axes[0].axvline(0, color="white", linewidth=1, linestyle="--")
    axes[0].set_title(
        f"Ending Equity -- {MC_TRADE_COUNT} trades, {MC_SIMULATIONS:,} sims",
        color="white", fontsize=10, fontweight="bold"
    )
    axes[0].set_xlabel("P&L ($)", color="#aaa")
    axes[0].set_ylabel("Density", color="#aaa")
    axes[0].legend(fontsize=6.5, labelcolor="white",
                   facecolor="#1a1a2e", edgecolor="#333")
    axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))

    axes[1].set_title("Max Drawdown Distribution",
                      color="white", fontsize=10, fontweight="bold")
    axes[1].set_xlabel("Max Drawdown ($)", color="#aaa")
    axes[1].set_ylabel("Density", color="#aaa")
    axes[1].legend(fontsize=6.5, labelcolor="white",
                   facecolor="#1a1a2e", edgecolor="#333")
    axes[1].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))

    fig.suptitle(
        f"Monte Carlo -- Top 6 Variants  (ruin threshold: ${MC_RUIN_THRESHOLD:,})",
        color="white", fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    path = os.path.join(output_dir, "orb_montecarlo.png")
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.close()
    print(f"   Saved: {path}")


# ─────────────────────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load
df = load_data(INPUT_FILE)

# 2. ORB table
orb_df = build_orb_table(df)

# 3. Breakout detection
orb_df = detect_breakouts(df, orb_df)

# 4. Backtest
trades_df = run_backtest(df, orb_df)

trades_path = os.path.join(OUTPUT_DIR, "orb_trades_full.csv")
trades_df.to_csv(trades_path, index=False)
print(f"Trade log saved: {trades_path}")

# 5. Summary
print("\n PERFORMANCE SUMMARY (sorted by expectancy):")
summary_df = summarise_results(trades_df)
print(summary_df[["variant", "n_trades", "win_rate", "profit_factor",
                   "expectancy_$", "total_pnl_$", "max_drawdown_$",
                   "max_single_loss_$", "max_consec_losses",
                   "avg_risk_pts"]].to_string(index=False))

summary_path = os.path.join(OUTPUT_DIR, "orb_results_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")

# 6. Monte Carlo (top 6 by expectancy)
print(f"\n Monte Carlo ({MC_SIMULATIONS:,} sims, {MC_TRADE_COUNT} trades, "
      f"ruin = ${MC_RUIN_THRESHOLD:,})...")

mc_results = []
for _, row in summary_df.head(6).iterrows():
    mc = run_monte_carlo(trades_df, row)
    if mc:
        mc_results.append(mc)
        print(f"   {mc['variant']}")
        print(f"     Median equity:  ${mc['median_end_equity']:>10,.0f}")
        print(f"     5th pct equity: ${mc['p5_end_equity']:>10,.0f}")
        print(f"     Median max DD:  ${mc['median_max_dd']:>10,.0f}")
        print(f"     Prob of ruin:    {mc['prob_ruin_%']:>8.1f}%")

mc_summary = pd.DataFrame([
    {k: v for k, v in mc.items()
     if k not in ["ending_equity_dist", "max_dd_dist"]}
    for mc in mc_results
])
mc_path = os.path.join(OUTPUT_DIR, "orb_montecarlo.csv")
mc_summary.to_csv(mc_path, index=False)
print(f"MC summary saved: {mc_path}")

# 7. Charts
print("\n Generating charts...")
plot_equity_curves(trades_df, summary_df, OUTPUT_DIR)
plot_monte_carlo(mc_results, OUTPUT_DIR)

print(f"\n Done. All files saved to: {OUTPUT_DIR}")


 Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
   Raw shape:  (4974698, 7)
   Columns:    ['Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume']
   Clean shape: (4974698, 10)
   Date range:  2010-06-06 to 2025-03-14

 ORB Table: 3789 days | Bull: 1931 | Bear: 1858 | Mean range: 25.8 pts
